# VGG
- Mạnh hơn nhờ việc giảm chậm độ phân giải ảnh
- chia thành các khối, mỗi khối gồm 1 hoặc nhiều lớp tích chập và 1 lớp max pooling
- học được nhiều hơn trước khi giảm độ phân giải ảnh
- mạng sâu hơn -> học tốt hơn

In [2]:
import torch
from torch import nn

In [6]:
class VGG(nn.Module):
    def __init__(self, num_class, info_block):
        super(VGG, self).__init__()
        blocks = []
        for num_chanels, num_conv in info_block:
            blocks.append(self.vgg_block(num_chanels, num_conv))
        self.net = nn.Sequential(
            *blocks,
            nn.Flatten(),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(p=0.5),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(p=0.5),
            nn.LazyLinear(num_class)   
        )
    def forward(self, X):
        return self.net(X)
    
    def vgg_block(self, num_ouput_channels, num_conv):
        block = []
        for i in range(num_conv):
            block.append(nn.LazyConv2d(out_channels=num_ouput_channels, kernel_size=3, padding=1))
            block.append(nn.ReLU())
        block.append(nn.MaxPool2d(kernel_size=2, stride=2))
        return nn.Sequential(*block)

    def layer_summary(self, X_shape):
        X = torch.randn(X_shape)
        for layer in self.net:
            X = layer(X)
            print(layer.__class__.__name__, 'output shape:\t', X.shape)


In [7]:
model = VGG(num_class=10, info_block=((64, 1), (128, 1), (256, 2), (512, 2), (512, 2)))
model.layer_summary((1, 1, 224, 224))

Sequential output shape:	 torch.Size([1, 64, 112, 112])
Sequential output shape:	 torch.Size([1, 128, 56, 56])
Sequential output shape:	 torch.Size([1, 256, 28, 28])
Sequential output shape:	 torch.Size([1, 512, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
Flatten output shape:	 torch.Size([1, 25088])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 10])


In [8]:
import torchvision
from torchvision import transforms

trans = torchvision.transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

data_train = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    transform=trans,
    download=True)
data_val = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    transform=trans,
    download=True
)

len(data_train), len(data_val)

(60000, 10000)

In [9]:
import torchvision
from torchvision import transforms

trans = torchvision.transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

data_train = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    transform=trans,
    download=True)
data_val = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    transform=trans,
    download=True
)

len(data_train), len(data_val)

(60000, 10000)

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    data_train,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    data_val,
    batch_size=128,
    shuffle=False
)

In [ ]:
from torch import optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = VGG(num_class=10, info_block=((64, 1), (128, 1), (256, 2), (512, 2), (512, 2))).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(params=model.parameters(), lr=0.01, weight_decay=0.01)

epochs = 10

loss_histoty = {"train": [], "val": []}

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_hat = model(X_batch)

        loss = criterion(y_hat, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()
    train_loss_avg = train_loss / len(train_loader)
    loss_histoty["train"].append(train_loss_avg)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_hat = model(X_batch)
            loss = criterion(y_hat, y_batch)
            val_loss += loss.item()
        
        val_loss_avg = val_loss / len(val_loader)
        loss_histoty["val"].append(val_loss_avg)
    print(f"Epoch {epoch+1}/{epochs}: Train Loss: {train_loss_avg}, Val Loss: {val_loss_avg}")


Using device: cuda
Epoch 1/10: Train Loss: 2.302674616323605, Val Loss: 2.3025722443302974
Epoch 2/10: Train Loss: 2.3026276148204357, Val Loss: 2.3025703068020977
Epoch 3/10: Train Loss: 2.302615585103472, Val Loss: 2.302574015870879


: 

In [ ]:
from matplotlib import pyplot as plt

plt.plot(loss_histoty["train"], label="train_loss")
plt.plot(loss_histoty["val"], label = "val_loss")

plt.show()

In [ ]:
from sklearn.metrics import classification_report

# Tính toán predictions trên tất cả validation data
y_true = []
y_pred = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_hat = model.forward(X_batch)
        predictions = y_hat.argmax(dim=1)
        
        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

print(classification_report(y_true, y_pred))